In [1]:
from anthropic import Anthropic
from dotenv import load_dotenv
import os

In [2]:
load_dotenv(override=True)

client = Anthropic()
model = "claude-sonnet-5"

# Some helpful error return types:
 - transient: a temporary system error like a timeout (retryable)
 - validation: the input is malformed or incomplete (not retryable)
 - business: understood, but business rule disallows (not retryable)
 - permisson: the user isn't allowed access or action (not retryable)

In [3]:
def lookup_order_by_id(order_id: str) -> dict:
    if not order_id.startswith("ORD-"):
        return {
            "isError": True,
            "errorCategory": "validation",
            "isRetryable": False,
            "customerMessage": "The order ID does not look valid.  Please check the order number and try again.",
            "developerMessage": "order ID must start with ORD-."
        }

    if order_id == "ORD-0000000":
        return {
            "isError": False,
            "orders": []
        }

    if order_id == "ORD-9999999":
        return {
            "isError": True,
            "errorCategory": "permission",
            "isRetryable": False,
            "customerMessage": "Ican't access this order with the current account information.",
            "developerMessage": "Authenticated customer does not own the requested order."
        }

    if order_id == "ORD-TIMEOUT":
        return {
            "isError": True,
            "errorCategory": "transient",
            "isRetryable": True,
            "customerMessage": "I'm having trouble checking the order right now.  Please try again later.",
            "developerMessage": "Order service timed out."
        }

    return {
        "isError": False,
        "order": {
            "order_id": order_id,
            "status": "delivered",
            "delivered_days_ago": 42,
            "total": 89.99
        }
    }

In [4]:
def check_refund_eligibility(order: dict) -> dict:
    if order["delivered_days_ago"] > 30:
        return {
            "isError": True,
            "errorCategory": "business",
            "isRetryable": False,
            "customerMessage": "This order is outside the standard return window.  I cannot process this return automatically.",
            "developerMessage": "Refund denied because the order is outside the 30 day return window."
        }

    return {
        "isError": False,
        "eligible": True,
        "reason": "Order is within the 30 day return window."
    }

# Escalate only after local recovery fails
- tool error: transient -> retry locally, messy input -> normalize input data, missing information -> ask user
- if resolved, continue - if not, then escalate